# DQ Framework vs Great Expectations – Benchmark

Τρέχει τους ίδιους κανόνες (`is_not_null` στο `l_shipdate`, `check_thresholds_dynamic` στο `l_quantity`) στο `analytics_dq_dev.benchmark.lineitem_dirty` με τα δύο frameworks και γράφει τους χρόνους στο `analytics_dq_dev.benchmark.comparison_results`.

Προαπαιτούμενα: το `../setup/results_table_and_views` έχει τρέξει μία φορά. Τα views `report_inside_outside` και `report_rule_by_rule` ανανεώνονται αυτόματα με κάθε νέο run.

In [0]:
%pip install databricks-labs-dqx -q

In [0]:
dbutils.library.restartPython()

In [0]:
import time
import uuid
from datetime import datetime
from pyspark.sql import functions as F
import great_expectations as gx
from great_expectations.dataset import SparkDFDataset

CATALOG       = "analytics_dq_dev"
BENCHMARK_TBL = f"{CATALOG}.benchmark.comparison_results"
DATASET_NAME  = "analytics_dq_dev.benchmark.lineitem_dirty"
RUN_ID        = str(uuid.uuid4())
CLUSTER_ID    = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "unknown")

print(f"🚀 Benchmark Run ID: {RUN_ID}")
print(f"📦 Dataset         : {DATASET_NAME}")
print(f"🖥️  Cluster         : {CLUSTER_ID}")

In [0]:
df_lineitem = spark.table(DATASET_NAME).cache()
row_count   = df_lineitem.count()   # force cache & materialize

print(f"✅ Dataset loaded: {row_count:,} rows")

In [0]:
def write_benchmark_result(framework, rule_name, column_tested,
                            rows_evaluated, rows_passed, rows_failed,
                            rule_status, execution_time_sec, total_time_sec):
    data = [{
        "run_id":             RUN_ID,
        "run_timestamp":      datetime.utcnow(),
        "framework":          framework,
        "dataset":            DATASET_NAME,
        "row_count":          row_count,
        "rule_name":          rule_name,
        "column_tested":      column_tested,
        "rows_evaluated":     rows_evaluated,
        "rows_passed":        rows_passed,
        "rows_failed":        rows_failed,
        "rule_status":        rule_status,
        "execution_time_sec": round(execution_time_sec, 4),
        "cluster_id":         CLUSTER_ID,
        "total_time_sec":     round(total_time_sec, 4)
    }]
    spark.createDataFrame(data).write.format("delta").mode("append").saveAsTable(BENCHMARK_TBL)
    print(f"  💾 Saved: [{framework}] {rule_name} on {column_tested} → {execution_time_sec:.2f}s")

In [0]:
import sys
import importlib
sys.path.append('/Workspace/Users/leonidas.avgoustinos@ms.d-one.ai/data-quality-framework')
import framework.utils, framework.dq_framework
importlib.reload(framework.utils)
importlib.reload(framework.dq_framework)
from framework.dq_framework import run_data_quality

print("=" * 55)
print("  DQ FRAMEWORK – Running...")
print("=" * 55)

t_start = time.time()

result_id = run_data_quality(
    apply_at="gold",
    run_mode="all",
    project_name="demo_project",
    full_table_name=DATASET_NAME,
    display_logs=False,
    full_scan=True
)

# Διάβασμα αποτελεσμάτων από τον result_rule πίνακα
df_results = (
    spark.table("analytics_dq_dev.results.result_rule")
    .filter(F.col("result_id") == result_id)
    .join(
        spark.table("analytics_dq_dev.configuration.rule_assignment").select("rule_assignment_id", "template_nk", "parameters_identifiers"),
        on="rule_assignment_id"
    )
    .select("template_nk", "parameters_identifiers", "rows_evaluated", "rows_pass", "rows_failed", "status", "start_timestamp", "end_timestamp")
)

# Γράφουμε ένα row ανά κανόνα
results_rows = df_results.collect()
t_total = time.time() - t_start
print(f"\n✅ DQ Framework total time: {t_total:.2f}s")

for row in results_rows:
    col_info  = row["parameters_identifiers"] or ""
    exec_secs = (row["end_timestamp"] - row["start_timestamp"]).total_seconds()
    write_benchmark_result(
        framework       = "dq_framework",
        rule_name       = row["template_nk"],
        column_tested   = col_info,
        rows_evaluated  = row["rows_evaluated"] or 0,
        rows_passed     = row["rows_pass"] or 0,
        rows_failed     = row["rows_failed"] or 0,
        rule_status     = row["status"],
        execution_time_sec = exec_secs,
        total_time_sec  = t_total
    )

print("✅ DQ Framework results saved!")

In [0]:
print("=" * 55)
print("  GREAT EXPECTATIONS – Running...")
print("=" * 55)

t_ge_start = time.time()
ge_df = SparkDFDataset(df_lineitem)

# ------- Rule 1: is_not_null on l_shipdate -------
t0 = time.time()
result_null = ge_df.expect_column_values_to_not_be_null("l_shipdate")
t1 = time.time()

r = result_null["result"]
rows_eval   = r.get("element_count", row_count)
rows_failed = r.get("unexpected_count", 0)
rows_passed = rows_eval - rows_failed
status      = "pass" if result_null["success"] else "fail"

write_benchmark_result(
    framework       = "great_expectations",
    rule_name       = "is_not_null",
    column_tested   = '{"column": "l_shipdate"}',
    rows_evaluated  = rows_eval,
    rows_passed     = rows_passed,
    rows_failed     = rows_failed,
    rule_status     = status,
    execution_time_sec = t1 - t0,
    total_time_sec  = 0.0  # placeholder, updated below
)

# ------- Rule 2: l_quantity between 0 and 999999 -------
t0 = time.time()
result_gt = ge_df.expect_column_values_to_be_between("l_quantity", min_value=0, max_value=999999)
t1 = time.time()

r = result_gt["result"]
rows_eval   = r.get("element_count", row_count)
rows_failed = r.get("unexpected_count", 0)
rows_passed = rows_eval - rows_failed
status      = "pass" if result_gt["success"] else "fail"

write_benchmark_result(
    framework       = "great_expectations",
    rule_name       = "check_thresholds_dynamic",
    column_tested   = '{"column": "l_quantity"}',
    rows_evaluated  = rows_eval,
    rows_passed     = rows_passed,
    rows_failed     = rows_failed,
    rule_status     = status,
    execution_time_sec = t1 - t0,
    total_time_sec  = 0.0  # placeholder, updated below
)

t_ge_total = time.time() - t_ge_start
print(f"\n✅ Great Expectations total time: {t_ge_total:.2f}s")

# Update total_time_sec for GE rows
from delta.tables import DeltaTable
dt = DeltaTable.forName(spark, BENCHMARK_TBL)
dt.update(
    condition=(F.col("run_id") == RUN_ID) & (F.col("framework") == "great_expectations"),
    set={"total_time_sec": F.lit(round(t_ge_total, 4))}
)
print("✅ Great Expectations results saved!")

In [0]:
display(
    spark.table(BENCHMARK_TBL)
    .filter(F.col("run_id") == RUN_ID)
    .select("framework", "rule_name", "column_tested",
            "rows_evaluated", "rows_failed", "rule_status", "execution_time_sec", "total_time_sec")
    .orderBy("rule_name", "framework")
)

In [0]:
%sql
SELECT exec_label, data_volume, metric, gx_sec, dq_sec, dq_vs_gx
FROM analytics_dq_dev.benchmark.report_inside_outside
ORDER BY exec_no, metric_order

In [0]:
%sql
SELECT exec_label, data_volume, metric, gx_sec, dq_sec, dq_vs_gx, dq_rules_batched
FROM analytics_dq_dev.benchmark.report_rule_by_rule
ORDER BY exec_no, metric